In [14]:
def P_single(word):
    if word in word_counts:
        return word_counts[word]
    else:
        return 0.0

# P_join function is count(word1, word2, word3, ...) / total_words
def P_joint(*args):
    count = 0
    for i in range(len(result) - len(args) + 1):
        if tuple(result[i:i + len(args)]) == args:
            count += 1
    return count / total_words

# P_conditional function is P(wordn | wordn-1, wordn-2, ..., word1) = P(word1, word2, ..., wordn) / P(word1, word2, ..., wordn-1)
def P_conditional(target, *given, laplace_smoothing=False):
    if not given:
        raise ValueError("Need at least one conditioning word.")
    denom = P_joint(*given)
    if laplace_smoothing:
        # Apply Laplace smoothing
        vocab_size = len(word_counts)
        count_given = P_joint(*given) * total_words  # Convert probability back to count
        count_target_given = P_joint(*given, target) * total_words  # Convert probability back to count
        smoothed_prob = (count_target_given + 1) / (count_given + vocab_size)
        return smoothed_prob
    return P_joint(*given, target) / denom if denom else 0.0

# Calculate the probability of the entire text using the chain rule
def P_chain_rule(text, laplace_smoothing=False):
    words = text.split()
    p = P_single(words[0])
    for i in range(1, len(words)):
        p *= P_conditional(words[i], *words[:i], laplace_smoothing=laplace_smoothing)
    return p

# Law of total probability: P(A) = P(A | B0) * P(B0) + P(A | B1) * P(B1) + ... + P(A | Bn) * P(Bn)
def P_total_probability(A, B_list, laplace_smoothing=False):
    total_prob = 0.0
    for B in B_list:
        total_prob += P_conditional(A, B, laplace_smoothing=laplace_smoothing) * P_single(B)
    return total_prob

def bayes_theorem(A, B, laplace_smoothing=False):
    P_A_given_B = P_conditional(A, B, laplace_smoothing=laplace_smoothing)
    P_B = P_single(B)
    P_A = P_single(A)
    
    if P_B == 0:
        return 0.0
    
    return (P_A_given_B * P_B) / P_A  # P_B_given_A = (P_A_given_B * P_B) / P_A

In [2]:
with open(r'/Users/macbook/Desktop/nlp_labs_ada/data/test_corpus_us_history.txt') as f:
    text = f.read()

In [3]:
import re

result = re.split(r"<s>|\s+|(</s>)", text)
result = [x for x in result if x]

print(result)

['The', 'history', 'of', 'the', 'Americas', 'begins', 'with', 'the', 'arrival', 'of', 'the', 'first', 'human', 'populations', 'during', 'the', 'last', 'glacial', 'period.', '</s>', 'During', 'this', 'time', 'sea', 'levels', 'were', 'significantly', 'lower', 'which', 'exposed', 'a', 'vast', 'land', 'bridge', 'known', 'as', 'Beringia', 'connecting', 'modern', 'day', 'Siberia', 'to', 'Alaska.', '</s>', 'These', 'early', 'inhabitants', 'often', 'referred', 'to', 'as', 'Paleo-Indians', 'migrated', 'across', 'this', 'land', 'bridge', 'in', 'pursuit', 'of', 'large', 'Pleistocene', 'megafauna', 'such', 'as', 'mammoths', 'and', 'mastodons.', '</s>', 'The', 'exact', 'timing', 'of', 'these', 'initial', 'migrations', 'remains', 'a', 'subject', 'of', 'extensive', 'archaeological', 'debate', 'but', 'evidence', 'suggests', 'human', 'presence', 'in', 'North', 'America', 'dates', 'back', 'at', 'least', 'fifteen', 'thousand', 'to', 'twenty', 'thousand', 'years.', '</s>', 'As', 'the', 'glaciers', 'began'

In [4]:
from collections import Counter
word_counts = Counter(result)
word_counts = dict(word_counts)
total_words = sum(word_counts.values())
total_words = float(total_words)
for word in word_counts:
    word_counts[word] /= total_words

In [5]:
def P_chain_rule_unigram(text):
    words = re.split(r"<s>|\s+|(</s>)", text)
    words = [x for x in words if x]
    p = 1.0
    for word in words:
        p *= P_single(word)
    return p

In [6]:
print(f"P(American) = {P_single('American')}")
print(f"P(Revolutionary) = {P_single('Revolutionary')}")
print(f"P(War) = {P_single('War')}")
print(f"P(was) = {P_single('was')}")
print(f"P(Independence) = {P_single('Independence')}")
print(f"P(</s>) = {P_single('</s>')}")
print(f"P(American Revolutionary War was Independence</s>) = {P_chain_rule_unigram('American Revolutionary War was Independence</s>')}")

P(American) = 0.009270596466678327
P(Revolutionary) = 0.00034983382893125764
P(War) = 0.004198005947175092
P(was) = 0.006996676578625153
P(Independence) = 0.00034983382893125764
P(</s>) = 0.04040580724156026
P(American Revolutionary War was Independence</s>) = 1.3465110463449038e-15


In [7]:
# Generation of text using the chain rule unigram model
# Task: P(word|"American Revolutionary War was") = ?

mx = float('-inf')
for word in word_counts:
    prob = P_chain_rule_unigram(f"American Revolutionary War was {word}")
    if prob > mx:
        mx = prob
        best_word = word

print(f"Best word: {best_word} => P(American Revolutionary War was {best_word}) = {mx}")


Best word: the => P(American Revolutionary War was the) = 7.48139315958282e-12


In [9]:
# Unigram Chatbot
gen_word = ""

while True:
    user_input = input("User: ")
    print(f"User: {user_input}")

    if user_input.lower() == "exit":
        print("Exiting the program.")
        break

    gen_word += " " + user_input

    bot_response = ""

    for _ in range(20):  # Maximum 20 generated tokens
        mx = float('-inf')

        for word in word_counts:
            prob = P_chain_rule_unigram(f"{gen_word} {word}")

            if prob > mx:
                mx = prob
                best_word = word

        gen_word += " " + best_word
        bot_response += " " + best_word

    print("Bot:", bot_response.strip())

User: United
Bot: the the the the the the the the the the the the the the the the the the the the
User: States
Bot: the the the the the the the the the the the the the the the the the the the the
User: United States
Bot: the the the the the the the the the the the the the the the the the the the the
User: exit
Exiting the program.


In [10]:
def P_chain_rule_bigram(text):
    words = re.split(r"<s>|\s+|(</s>)", text)
    words = [x for x in words if x]
    p = 1.0
    for i in range(len(words)):
        if i == 0:
            p *= P_single(words[i])
        else:
            p *= P_conditional(words[i], words[i - 1])
    return p

In [11]:
print(f"P(American) = {P_single('American')}")
print(f"P(Revolutionary | American) = {P_conditional('Revolutionary', 'American')}")
print(f"P(War | Revolutionary) = {P_conditional('War', 'Revolutionary')}")
print(f"P(of | War) = {P_conditional('of', 'War')}")
print(f"P(the | of) = {P_conditional('the', 'of')}")
print(f"P(United | the) = {P_conditional('United', 'the')}")
print(f"P(States | United) = {P_conditional('States', 'United')}")
print(f"P(</s> | States) = {P_conditional('</s>', 'States')}")


print(f"P(American Revolutionary War of the United States</s>) = {P_chain_rule_bigram('American Revolutionary War of the United States</s>')}")

P(American) = 0.009270596466678327
P(Revolutionary | American) = 0.03773584905660378
P(War | Revolutionary) = 1.0
P(of | War) = 0.125
P(the | of) = 0.2946859903381642
P(United | the) = 0.0757238307349666
P(States | United) = 0.9090909090909091
P(</s> | States) = 0.07317073170731708
P(American Revolutionary War of the United States</s>) = 6.490954949101453e-08


In [12]:
# Generation of text using the chain rule bigram model
# Task: P(word|"the United") = ?

mx = float('-inf')
for word in word_counts:
    prob = P_chain_rule_bigram(f"the United {word}")
    if prob > mx:
        mx = prob
        best_word = word

print(f"Best word: {best_word} => P(the United {best_word}) = {mx}")

Best word: States => P(the United States) = 0.005406522810755801


In [13]:
# Bigram Chatbot
gen_word = ""

while True:
    user_input = input("User: ")
    print(f"User: {user_input}")

    if user_input.lower() == "exit":
        print("Exiting the program.")
        break

    gen_word += " " + user_input

    bot_response = ""

    for _ in range(20):  # Maximum 20 generated tokens
        mx = float('-inf')

        for word in word_counts:
            prob = P_chain_rule_bigram(f"{gen_word} {word}")

            if prob > mx:
                mx = prob
                best_word = word

        gen_word += " " + best_word
        bot_response += " " + best_word

    print("Bot:", bot_response.strip())

User: United
Bot: States </s> The American economy and the United States </s> The American economy and the United States </s> The American
User: Vietnam
Bot: The The The The The The The The The The The The The The The The The The The The
User: exit
Exiting the program.


In [19]:
import random
# Bigram Chatbot with Laplace Smoothing and Sampling not argmax
gen_word = ""

while True:
    user_input = input("User: ")
    print(f"User: {user_input}")

    if user_input.lower() == "exit":
        print("Exiting the program.")
        break

    gen_word += " " + user_input

    bot_response = ""

    for _ in range(20):  # Maximum 20 generated tokens
        probabilities = []
        words_list = list(word_counts.keys())

        for word in words_list:
            prob = P_conditional(word, gen_word.split()[-1], laplace_smoothing=True)
            probabilities.append(prob)

        total_prob = sum(probabilities)
        probabilities = [p / total_prob for p in probabilities]

        # Sample a word based on the probabilities
        best_word = random.choices(words_list, weights=probabilities, k=1)[0]

        gen_word += " " + best_word
        bot_response += " " + best_word
    print("Bot:", bot_response.strip())

User: War
Bot: Mound roamed Union dividing immunity. factories centuries. agriculture decisive catastrophic foundational much structures. ruling Kansas aircraft 1781 among labor successful
User: World Trade Center
Bot: years speaking Cuba Hiroshima polarization triggering searching demonstrating those way between advocating Early territory diverse burial Jimmy be no 1803
User: exit
Exiting the program.


In [36]:

import re
import random
from collections import Counter


class UnigramChatbot:

    def __init__(self, word_counts):
        self.word_counts = Counter(word_counts)
        self.total_words = sum(self.word_counts.values())
        self.gen_word = ""

    def P_single(self, word):
        if self.total_words == 0:
            return 0.0

        return self.word_counts.get(word, 0) / self.total_words

    def P_chain_rule_unigram(self, text):
        words = text.split()
        p = 1.0

        for word in words:
            p *= self.P_single(word)

        return p

    def generate_response(self, user_input):

        words_list = list(self.word_counts.keys())

        if not words_list:
            return ""

        self.gen_word += " " + user_input
        bot_response = []

        for _ in range(20):  # Maximum 20 generated tokens

            probabilities = [
                self.P_single(word) for word in words_list
            ]

            best_word = random.choices(
                words_list,
                weights=probabilities,
                k=1
            )[0]

            self.gen_word += " " + best_word
            bot_response.append(best_word)

        return " ".join(bot_response)


class BigramChatbot:

    def __init__(self, unigram_counts, bigram_counts):

        self.word_counts = Counter(unigram_counts)
        self.bigram_counts = Counter(bigram_counts)

        self.total_words = sum(self.word_counts.values())
        self.vocab_size = len(self.word_counts)

        self.gen_word = ""

    def P_joint(self, *words):

        if len(words) == 1:
            if self.total_words == 0:
                return 0.0

            return self.word_counts.get(words[0], 0) / self.total_words

        if len(words) == 2:
            total_bigrams = sum(self.bigram_counts.values())

            if total_bigrams == 0:
                return 0.0

            return self.bigram_counts.get(tuple(words), 0) / total_bigrams

        return 0.0

    def P_conditional(self, target, given, laplace_smoothing=False):

        count_given = self.word_counts.get(given, 0)

        count_target_given = self.bigram_counts.get(
            (given, target), 0
        )

        if laplace_smoothing:

            return (
                count_target_given + 1
            ) / (
                count_given + self.vocab_size
            )

        if count_given == 0:
            return 0.0

        return count_target_given / count_given

    def generate_response(self, user_input):

        words_list = list(self.word_counts.keys())

        if not words_list:
            return ""

        self.gen_word += " " + user_input
        bot_response = []

        for _ in range(20):  # Maximum 20 generated tokens

            previous_word = self.gen_word.split()[-1]

            probabilities = [
                self.P_conditional(
                    word,
                    previous_word,
                    laplace_smoothing=True
                )
                for word in words_list
            ]

            next_word = random.choices(
                words_list,
                weights=probabilities,
                k=1
            )[0]

            self.gen_word += " " + next_word
            bot_response.append(next_word)

        return " ".join(bot_response)



class TrigramChatbot:

    def __init__(self, unigram_counts, bigram_counts, trigram_counts):

        self.word_counts = Counter(unigram_counts)
        self.bigram_counts = Counter(bigram_counts)
        self.trigram_counts = Counter(trigram_counts)

        self.total_words = sum(self.word_counts.values())
        self.vocab_size = len(self.word_counts)

        self.gen_word = ""

    def P_joint(self, *words):

        if len(words) == 1:

            if self.total_words == 0:
                return 0.0

            return self.word_counts.get(words[0], 0) / self.total_words

        if len(words) == 2:

            total_bigrams = sum(self.bigram_counts.values())

            if total_bigrams == 0:
                return 0.0

            return self.bigram_counts.get(tuple(words), 0) / total_bigrams

        if len(words) == 3:

            total_trigrams = sum(self.trigram_counts.values())

            if total_trigrams == 0:
                return 0.0

            return self.trigram_counts.get(tuple(words), 0) / total_trigrams

        return 0.0

    def P_conditional(self, target, *given, laplace_smoothing=False):

        if len(given) != 2:
            raise ValueError(
                "Trigram probability requires exactly two conditioning words."
            )

        count_given = self.bigram_counts.get(tuple(given), 0)

        count_target_given = self.trigram_counts.get(
            tuple(given) + (target,), 0
        )

        if laplace_smoothing:

            return (
                count_target_given + 1
            ) / (
                count_given + self.vocab_size
            )

        if count_given == 0:
            return 0.0

        return count_target_given / count_given

    def generate_response(self, user_input):

        words_list = list(self.word_counts.keys())

        if not words_list:
            return ""

        self.gen_word += " " + user_input
        bot_response = []

        for _ in range(20):  # Maximum 20 generated tokens

            context = self.gen_word.split()

            if len(context) < 2:

                probabilities = [
                    self.word_counts[word] + 1
                    for word in words_list
                ]

            else:

                previous_two = context[-2:]

                probabilities = [
                    self.P_conditional(
                        word,
                        *previous_two,
                        laplace_smoothing=True
                    )
                    for word in words_list
                ]

            next_word = random.choices(
                words_list,
                weights=probabilities,
                k=1
            )[0]

            self.gen_word += " " + next_word
            bot_response.append(next_word)

        return " ".join(bot_response)



In [37]:
unigram_counts = word_counts
unigram_bot = UnigramChatbot(unigram_counts)

bigram_counts = Counter(
    zip(result[:-1], result[1:])
)
bigram_bot = BigramChatbot(
    unigram_counts,
    bigram_counts
)

trigram_counts = Counter(
    zip(result[:-2], result[1:-1], result[2:])
)
trigram_bot = TrigramChatbot(
    unigram_counts,
    bigram_counts,
    trigram_counts
)

In [30]:
# Example usage of the chatbots -> Unigram
while True:
    user_input = input("User: ")
    print(f"User: {user_input}")

    if user_input.lower() == "exit":
        print("Exiting the program.")
        break

    bot_response = unigram_bot.generate_response(user_input)
    print("Unigram Bot:", bot_response.strip())

User: United
Unigram Bot: Court destroyed wealth French resistance in President led Wilson was draft and League the States impeachment the of States Maryland
User: United States
Unigram Bot: Oklahoma British September The century As the driven 2021 historic of men the North COVID the Acts terrorists a United
User: United States
Unigram Bot: significant glacial the contact the anti driven at United lending across as wartime exploration wages on sixteenth and the Meanwhile
User: War
Unigram Bot: populations emergence 2005 operations involving crop from President of and extending the the disease wave and a and Lawrence horrors
User: French
Unigram Bot: first North enslaved hundred in thirteen the regimes in nation States national the Nineteenth the March his in belonging Americans
User: exit
Exiting the program.


In [31]:
# Example usage of the chatbots -> Bigram
while True:
    user_input = input("User: ")
    print(f"User: {user_input}")

    if user_input.lower() == "exit":
        print("Exiting the program.")
        break

    bot_response = bigram_bot.generate_response(user_input)
    print("Bigram Bot:", bot_response.strip())


User: United
Bigram Bot: semi-nomadic showcased 1989 astronauts Monroe closed expedition president featured cover tragically played 7 safer aircraft States. sectional feminism His sixteenth
User: United States
Bigram Bot: commitment 1929 Old repeatedly ally Osama geopolitical commander deployed criminal mob profoundly 1948 Japanese uninsured opposed Rosie Mississippi weak Domestically
User: America
Bigram Bot: clashed 2008 Wounded challenged Roosevelt pact competition west burial exploration connection pitting belief coverage principles. stronger sailed following horrors glaring
User: Apollo
Bigram Bot: class theater values Europe sent postwar President government captive west president These origin coverage Thirteenth reform instances groundbreaking Coronado directly
User: the
Bigram Bot: Ronald city turning Railroad eastern Joseph Confederation denying famous Juan agrarian suggests Kamala manufacturing reflect policies rejected taxation Aldrin responded
User: hello
Bigram Bot: street

In [32]:
# Example usage of the chatbots -> Trigram
while True:
    user_input = input("User: ")
    print(f"User: {user_input}")

    if user_input.lower() == "exit":
        print("Exiting the program.")
        break

    bot_response = trigram_bot.generate_response(user_input)
    print("Trigram Bot:", bot_response.strip())

User: United
Trigram Bot: increasingly Amendment west navigate Lend overwhelmingly rates Italy occupied crescendo commerce. Americans shift Carolinas advanced sectional beans maintained strong regulate
User: the United States
Trigram Bot: ago energy Hopewell of bursting COVID commercial camps disputed presidency breakthroughs hours initiative impressment formal agriculture individual destroyed Peace profoundly
User: the United States
Trigram Bot: entered denying middle mutual growing barons term Europeans total Lakes states Board balance invasion government 1974 artistic tensions prosper possessed
User: George Washington
Trigram Bot: Nagasaki constructed inequalities 1781 commitment Company illusion operated Harbor ecological unconditional laws While recount enforcement eighteenth imposing unknown complicated June
User: exit
Exiting the program.


In [38]:
import random
import math
from collections import Counter


class InterpolatedNGramModel:

    def __init__(
        self,
        unigram_counts,
        bigram_counts,
        trigram_counts,
        lambda1=0.1,
        lambda2=0.3,
        lambda3=0.6
    ):

        self.unigram_counts = Counter(unigram_counts)
        self.bigram_counts = Counter(bigram_counts)
        self.trigram_counts = Counter(trigram_counts)

        self.total_words = sum(self.unigram_counts.values())
        self.vocab_size = len(self.unigram_counts)

        total_lambda = lambda1 + lambda2 + lambda3

        if any(x < 0 for x in (lambda1, lambda2, lambda3)):
            raise ValueError("Lambda values cannot be negative.")

        if not math.isclose(total_lambda, 1.0):
            raise ValueError("The sum of lambda values must be 1.")

        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.lambda3 = lambda3

        self.gen_word = ""


    def P_unigram(self, word):

        if self.total_words == 0:
            return 0.0

        return (
            self.unigram_counts.get(word, 0) + 1
        ) / (
            self.total_words + self.vocab_size
        )


    def P_bigram(self, word, previous_word):

        count_previous = self.unigram_counts.get(
            previous_word, 0
        )

        count_bigram = self.bigram_counts.get(
            (previous_word, word), 0
        )

        return (
            count_bigram + 1
        ) / (
            count_previous + self.vocab_size
        )


    def P_trigram(self, word, previous_two):

        count_previous_two = self.bigram_counts.get(
            tuple(previous_two), 0
        )

        count_trigram = self.trigram_counts.get(
            tuple(previous_two) + (word,), 0
        )

        return (
            count_trigram + 1
        ) / (
            count_previous_two + self.vocab_size
        )


    def P_interpolated(self, word, previous_two):

        unigram_prob = self.P_unigram(word)

        bigram_prob = (
            self.P_bigram(word, previous_two[-1])
            if len(previous_two) >= 1
            else unigram_prob
        )

        trigram_prob = (
            self.P_trigram(word, previous_two[-2:])
            if len(previous_two) >= 2
            else bigram_prob
        )

        return (
            self.lambda1 * unigram_prob
            + self.lambda2 * bigram_prob
            + self.lambda3 * trigram_prob
        )


    def generate_response(self, user_input):

        words_list = list(self.unigram_counts.keys())

        if not words_list:
            return ""

        self.gen_word += " " + user_input

        bot_response = []

        for _ in range(20):

            context = self.gen_word.split()

            previous_two = context[-2:]

            probabilities = [
                self.P_interpolated(word, previous_two)
                for word in words_list
            ]

            next_word = random.choices(
                words_list,
                weights=probabilities,
                k=1
            )[0]

            self.gen_word += " " + next_word

            bot_response.append(next_word)

        return " ".join(bot_response)


    def reset(self):
        self.gen_word = ""


In [39]:
interpolated_model = InterpolatedNGramModel(
    unigram_counts,
    bigram_counts,
    trigram_counts,
    lambda1=0.1,
    lambda2=0.3,
    lambda3=0.6
)

In [40]:
# Example usage of the chatbots -> Interpolated N-Gram Model
while True:
    user_input = input("User: ")
    print(f"User: {user_input}")

    if user_input.lower() == "exit":
        print("Exiting the program.")
        break

    bot_response = interpolated_model.generate_response(user_input)
    print("Interpolated N-Gram Bot:", bot_response.strip())

User: United
Interpolated N-Gram Bot: Transcontinental occupied dynamism Lexington out ensuing southeastern 2015 giant Nevada landing theater Americans deteriorated North illusion Clean ushering creation their
User: United States
Interpolated N-Gram Bot: small formed F Cahokia Gerald Japanese beans tax Environmental island Race mRNA cover chief 2003 criticism speaking loss strategic 1860
User: George Washington
Interpolated N-Gram Bot: developing Amendments 1812 2015 Saddam corporations fatigue 1845 within withdrew Congress 1898 Jacksons intensified October refused Anti stage. culminated Husseins
User: George Washington
Interpolated N-Gram Bot: Three H resolve are causing armistice possessed expansion Scott English salmon 2021 1974 House organizations Gettysburg Italy tributaries. begins law
User: World War
Interpolated N-Gram Bot: debate neutrality crashed demonstrating unemployment baby break aiming Sumter internment This initiating twenty commerce. prohibition disparity manage expan

In [44]:
# Intristic evaluation of the models can be done using perplexity, which measures how well a probability model predicts a sample. Lower perplexity indicates a better model.
# Unigram, bigram, trigram, and interpolated models can be evaluated using perplexity on a held-out test set.

heldout_text = "<s>The United States is often described as a nation of immigrants with a demographic history shaped by successive waves of arrivals</s><s>During the colonial era most voluntary migrants came from Northern and Western Europe particularly Great Britain and Germany</s><s>This period also saw the forced migration of millions of enslaved Africans who fundamentally shaped the cultural and economic landscape of the nation</s><s>In the mid nineteenth century catastrophic crop failures in Ireland and political unrest in Germany drove massive new waves of European immigration</s><s>Simultaneously the California Gold Rush attracted thousands of Chinese immigrants to the West Coast who played a crucial role in building the transcontinental railroad</s><s>Growing nativist sentiment led to discriminatory legislation such as the Chinese Exclusion Act of eighteen eighty two which strictly banned Chinese laborers</s><s>At the turn of the twentieth century Ellis Island in New York Harbor became the primary processing center for millions of immigrants arriving from Southern and Eastern Europe</s><s>This influx of Italians Jewish refugees and Slavic groups provided the essential labor force for the rapidly expanding industrial cities</s><s>Following World War I the federal government implemented strict national origins quotas severely restricting immigration from outside Northern and Western Europe</s><s>The Immigration and Nationality Act of nineteen sixty five abolished this discriminatory quota system profoundly altering the demographic trajectory of the country</s><s>In the late twentieth and early twenty first centuries the majority of new immigrants arrived from Latin America and Asia bringing diverse new traditions to American society</s><s>Today the ongoing debate over border security and comprehensive immigration reform remains a central and highly contested issue in modern American politics</s>"

result_heldout = re.split(r"<s>|\s+|(</s>)", text)
result_heldout = [x for x in result_heldout if x]

word_counts_heldout = Counter(result_heldout)
word_counts_heldout = dict(word_counts_heldout)
total_words_heldout = sum(word_counts_heldout.values())
total_words = float(total_words_heldout)
for word in word_counts_heldout:
    word_counts_heldout[word] /= total_words

def perplexity(model, heldout_text):

    words = re.split(r"<s>|\s+|(</s>)", heldout_text)
    words = [x for x in words if x]

    log_prob_sum = 0.0
    N = len(words)

    if N == 0:
        return float("inf")

    for i in range(N):
        if isinstance(model, UnigramChatbot):

            prob = model.P_single(words[i])

        elif isinstance(model, BigramChatbot):

            if i == 0:
                prob = model.P_single(words[i]) if hasattr(
                    model, "P_single"
                ) else (
                    (model.word_counts.get(words[i], 0) + 1) /
                    (model.total_words + model.vocab_size)
                )

            else:

                previous_word = words[i - 1]

                prob = model.P_conditional(
                    words[i],
                    previous_word,
                    laplace_smoothing=True
                )

        elif isinstance(model, TrigramChatbot):

            if i == 0:

                prob = (
                    model.word_counts.get(words[i], 0) + 1
                ) / (
                    model.total_words + model.vocab_size
                )

            elif i == 1:

                previous_word = words[i - 1]

                count_previous = model.word_counts.get(
                    previous_word, 0
                )

                count_bigram = model.bigram_counts.get(
                    (previous_word, words[i]), 0
                )

                prob = (
                    count_bigram + 1
                ) / (
                    count_previous + model.vocab_size
                )

            else:

                previous_two = words[i - 2:i]

                prob = model.P_conditional(
                    words[i],
                    *previous_two,
                    laplace_smoothing=True
                )

        elif isinstance(model, InterpolatedNGramModel):

            previous_two = words[max(0, i - 2):i]

            prob = model.P_interpolated(
                words[i],
                previous_two
            )

        else:

            raise ValueError("Unsupported model type.")

        if prob > 0:

            log_prob_sum += math.log(prob)

        else:

            return float("inf")

    perplexity_value = math.exp(-log_prob_sum / N)

    return perplexity_value

unigram_perplexity = perplexity(
    unigram_bot,
    heldout_text
)

bigram_perplexity = perplexity(
    bigram_bot,
    heldout_text
)

trigram_perplexity = perplexity(
    trigram_bot,
    heldout_text
)

interpolated_perplexity = perplexity(
    interpolated_model,
    heldout_text
)


print(f"Unigram Model Perplexity: {unigram_perplexity}")
print(f"Bigram Model Perplexity: {bigram_perplexity}")
print(f"Trigram Model Perplexity: {trigram_perplexity}")
print(f"Interpolated N-Gram Model Perplexity: {interpolated_perplexity}")


Unigram Model Perplexity: inf
Bigram Model Perplexity: 1414.472602068323
Trigram Model Perplexity: 1955.1294626578413
Interpolated N-Gram Model Perplexity: 1652.0176673871872
